# 6단계 · 정직한 F1과 모델 비교표 (평가 항목 'AI 예측모델' 40점)

지금까지 만든 점수를 **같은 규칙**으로 나란히 채점하고, 하나를 골라 그 근거를 적습니다.

**비교할 모델**
| 이름 | 종류 | 데이터 | 비고 |
|---|---|---|---|
| 무작위 | 기준선 | - | 1000번 평균 |
| RF / LR 간격0 | 지도학습 | 공식 | 누설이 있는 검증 — **참고용** |
| RF / LR 간격10 | 지도학습 | 공식 | 시험 행 ±10행을 학습에서 뺌 (누설 차단) |
| 이탈도 직전10 | 비지도 (라벨 안 씀) | 공식 | 과거만 봄 |
| 이탈도 전체기준 | 비지도 | 공식 | 미래 쇼트도 기준에 들어감 — **참고용** |
| 과거기준선 K10 | 비지도 | 공식 + 외부(원값·시각) | 과거만 봄, 비라벨 포함 |

질문 칸에는 **실행 전에 예상을 먼저** 적습니다.

In [ ]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from src.official import *
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

ML = pd.read_csv(RAW / "moldset_labeled.csv"); MLI = ML.set_index("Unnamed: 0")   # [외부] 원값·시각
DATA = {p: load(p)[0] for p in ["CN7", "RG3"]}
CEIL = {p: float(f1_ceiling(DATA[p])["F1"]) for p in DATA}
print({p: round(v, 3) for p, v in CEIL.items()})

### 6-1. 왜 지금까지의 'F1(최적, 참고)'을 보고하면 안 되나
`F1(최적)`은 **시험 데이터의 정답을 보면서** "여기서 자르면 F1이 가장 높다"는 기준점을 고른 값입니다.

> **질문 6-1.** 이게 4-1에서 본 '6개 중 최고 고르기'와 어떻게 같은 문제인가요?
>
> 내 답:

### 정직한 판정 규칙: 검사 예산
현장에서는 "하루에 몇 개까지 검사할 수 있다"가 먼저 정해집니다. 그래서 **결과를 보기 전에** 규칙을 정합니다.

> **점수가 높은 순서로 전체의 5%를 '불량'으로 판정한다** (보조: 10%)

이 규칙은 정답을 전혀 보지 않으므로 어느 모델에나 공정하게 적용됩니다.

> **질문 6-2.** CN7은 1,211행 중 불량 17개입니다. 5%면 61행을 '불량'으로 판정합니다. 완벽한 모델이라도 F1은 최대 얼마일까요?
> (힌트: 17개를 다 맞혀도 나머지 44개는 헛짚음. F1 = 2TP / (2TP + FP + FN))
>
> 내 답:

In [ ]:
def budget_f1(lab, s, q):
    """점수 상위 q 비율의 행을 '불량'으로 판정했을 때 (TP, FP, FN, F1). 동점은 무작위로 섞는다."""
    y = lab["y"].values
    tie = np.random.RandomState(0).rand(len(s))
    order = np.lexsort((tie, -np.asarray(s, float)))
    k = int(round(len(s) * q))
    pred = np.zeros(len(s), int); pred[order[:k]] = 1
    tp = int((pred & y).sum()); fp = k - tp; fn = int(y.sum()) - tp
    return tp, fp, fn, 2 * tp / (2 * tp + fp + fn)


def budget_ceiling(lab, q):
    """예산 q에서 낼 수 있는 최대 F1 (불량을 모두 예산 안에 넣었다고 가정)."""
    P, k = int(lab["y"].sum()), int(round(len(lab) * q))
    tp = min(P, k)
    return 2 * tp / (k + P)

for p, lab in DATA.items():
    print(p, {f"{int(q*100)}%": round(budget_ceiling(lab, q), 3) for q in [0.05, 0.10]})

### 모델 점수 만들기
3~5단계에서 쓴 함수를 그대로 가져왔습니다. 지도학습은 A_쌍층화 분할 + `gap_train_mask`(간격 두기)입니다.

In [ ]:
def robust_z(x, P):
    med = np.median(P, axis=0)
    sd = np.maximum(np.median(np.abs(P - med), axis=0) * 1.4826, P.std(axis=0))
    ok = sd > 1e-6
    return np.where(ok, np.minimum(np.abs(x - med) / np.where(ok, sd, 1), 20), 0)


def oof_supervised(lab, make_model, gap):
    """A_쌍층화 5-fold, 시험 행 ±gap 행을 학습에서 뺌. 학습할 불량이 없으면 0점."""
    X, y = lab[VARS].values, lab["y"].values
    fold = make_folds(lab, "A_쌍층화")
    s = np.zeros(len(lab))
    for k in range(K):
        te = fold == k
        tr = gap_train_mask(lab, te, gap)
        if y[tr].sum() == 0:
            continue
        s[te] = make_model().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
    return s


def deviation(lab, N=10, use_all=False):
    """4단계 이탈도. use_all=True 면 파일 전체 쇼트가 기준(미래 포함)."""
    sh = lab.drop_duplicates("pair"); X = sh[VARS].values
    sc = np.zeros(len(X))
    for i in range(len(X)):
        P = X if use_all else X[max(0, i - N): i]
        if len(P) >= 3:
            sc[i] = np.sqrt((robust_z(X[i], P) ** 2).mean())
    return lab["pair"].map(pd.Series(sc, index=sh["pair"].values)).values


def past_baseline(lab, p, Kb=10):
    """5단계 (나): 원값, 비라벨 정상 모드 포함 과거 Kb 쇼트 기준. [외부]"""
    ts = pd.to_datetime(lab["order"].map(MLI["TimeStamp"]))
    u = pd.read_csv(RAW / f"unlabeled_raw_{p.lower()}.csv"); u["ts"] = pd.to_datetime(u["TimeStamp"])
    u = u[u["Barrel_Temperature_1"] > 100].drop_duplicates("ts").sort_values("ts")
    first = ~lab["pair"].duplicated()
    X = MLI.loc[lab.loc[first, "order"], VARS].values
    bt, BX = u["ts"].values, u[VARS].values
    sc = [np.sqrt((robust_z(x, BX[bt < t][-Kb:]) ** 2).mean()) for x, t in zip(X, ts[first].values)]
    return lab["pair"].map(pd.Series(sc, index=lab.loc[first, "pair"].values)).values


RF = lambda: RandomForestClassifier(500, min_samples_leaf=3, class_weight="balanced_subsample", n_jobs=-1, random_state=0)
LR = lambda: LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000)

SCORES = {}
for p, lab in DATA.items():
    SCORES[p] = {
        "RF 간격0 (누설 참고)": oof_supervised(lab, RF, 0),
        "RF 간격10": oof_supervised(lab, RF, 10),
        "LR 간격0 (누설 참고)": oof_supervised(lab, LR, 0),
        "LR 간격10": oof_supervised(lab, LR, 10),
        "이탈도 직전10": deviation(lab),
        "이탈도 전체기준 (미래 포함, 참고)": deviation(lab, use_all=True),
        "과거기준선 K10 [외부]": past_baseline(lab, p),
    }
print("완료")

> **질문 6-3.** 정직한 규칙(예산 5%)으로 채점하면 CN7에서 1등은 어느 모델일까요? 지도학습(간격10)은 무작위보다 나을까요?
>
> 내 예상:

In [ ]:
rows = []
for p, lab in DATA.items():
    # 무작위: 1000번 평균
    rnd = [np.random.RandomState(s).rand(len(lab)) for s in range(1000)]
    r = {"모델": "무작위 (1000회 평균)", "제품": p,
         "AP": np.mean([evaluate(lab, x, "-", p, CEIL[p])["AP"] for x in rnd[:200]]), "유의": "-"}
    for q in [0.05, 0.10]:
        res = np.array([budget_f1(lab, x, q) for x in rnd])
        r[f"F1@{int(q*100)}%"] = res[:, 3].mean(); r[f"TP@{int(q*100)}%"] = res[:, 0].mean()
    rows.append(r)
    for name, s in SCORES[p].items():
        e = evaluate(lab, s, name, p, CEIL[p])
        r = {"모델": name, "제품": p, "AP": e["AP"], "유의": e["유의"]}
        for q in [0.05, 0.10]:
            tp, fp, fn, f1 = budget_f1(lab, s, q)
            r[f"F1@{int(q*100)}%"] = f1; r[f"TP@{int(q*100)}%"] = tp
        rows.append(r)

T6 = pd.DataFrame(rows)
for p, lab in DATA.items():
    T6.loc[T6["제품"] == p, "예산5% 상한 대비"] = T6["F1@5%"] / budget_ceiling(lab, 0.05)
    T6.loc[T6["제품"] == p, "정보 상한 대비"] = T6["F1@5%"] / CEIL[p]
T6.round(3)

### 검사 곡선: 검사량을 늘리면 불량을 몇 % 잡나 (현장활용 그림)
가로축 = 검사한 비율, 세로축 = 잡은 불량 비율. 대각선이 무작위입니다.

In [ ]:
import matplotlib.pyplot as plt
from src.plotting import setup, save
setup()

SHOW = ["RF 간격10", "LR 간격10", "이탈도 직전10", "과거기준선 K10 [외부]"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (p, lab) in zip(axes, DATA.items()):
    y = lab["y"].values
    for name in SHOW:
        s = SCORES[p][name]
        tie = np.random.RandomState(0).rand(len(s))
        o = np.lexsort((tie, -s))
        ax.plot(np.arange(1, len(s) + 1) / len(s), np.cumsum(y[o]) / y.sum(), label=name)
    ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="무작위")
    ax.axvline(0.05, color="gray", lw=0.8)
    ax.set_xlim(0, 0.3); ax.set_title(f"{p} 검사 곡선 (0~30%)")
    ax.set_xlabel("검사한 비율"); ax.set_ylabel("잡은 불량 비율"); ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
save(fig, "1002_검사곡선_모델비교")
plt.show()

> **질문 6-4.** 표와 그림을 보고 **선정 모델과 근거**를 3줄로 적어 보세요. 아래 기준을 하나씩 확인하면 됩니다.
> 1. 누설 없이 무작위보다 나은가? (유의, 간격10)
> 2. 현장에서 실시간으로 쓸 수 있나? (미래를 보지 않나)
> 3. 라벨이 거의 없어도 되나? (불량 17개, 25개)
> 4. 무엇을 잡고 무엇을 못 잡나? (4단계: 둘다불량 vs 쌍둥이)
>
> 내 답:

> **질문 6-5.** RG3는 어떤 모델도 무작위와 비슷합니다. 보고서에 이걸 '실패'로 쓸까요, '증명'으로 쓸까요? 1단계 정보 상한과 연결해 보세요.
>
> 내 답: